# A Lightweight U-Net–MobileNetV3-Small Framework for Portable Photoacoustic Imaging-Based Tumour-Associated Vascular Abnormality Detection

This notebook implements the complete pipeline with two databases kept separate at the data level:

**Database 1 — PAT public data:** preprocessing → portable degradation → lightweight U-Net reconstruction → reconstructed PAI images.

**Database 2 — Mendeley Photoacoustic Vascular Image Dataset:** original PAI images → VAE denoising → ROI extraction.

The two databases are **not row-wise merged**. Cross-database fusion is implemented by learning a Database-1 feature context from the training partition and fusing that context with each Database-2 sample through a shared MobileNetV3-Small feature extractor, coordinate attention and a 1×1 compression layer. This avoids inventing sample-to-sample correspondence between unrelated datasets.

The final classifier is trained and evaluated on the labelled Mendeley normal/tumour vascular images. Database 1 contributes reconstruction and cross-database representation learning rather than artificial tumour labels.

## Important dataset note

The PAT public repository provides sparse-view PAT data and describes a 180-view acquisition with 50,000 samples per acquisition and a 500 MHz sampling rate. Its public files include sparse 45-, 90- and 180-view MAT files. citeturn3view0

The Mendeley Version 2 dataset contains normal-vessel and tumour-vessel OR-PAM data; the published description reports 69 normal-vessel images, with 48 for training/validation and 21 for testing. citeturn1view0turn4search3

Because the two databases do not provide a common patient/image identifier, this notebook does **not** concatenate arbitrary images from the two datasets as if they were paired. That would create an invalid experimental design.

In [ ]:
!pip -q install scipy scikit-image opencv-python-headless tensorflow==2.16.1 tensorflow-model-optimization h5py pillow pandas matplotlib seaborn scikit-learn

In [ ]:
import os, glob, json, random, shutil, warnings, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import scipy.io as sio
from PIL import Image
from scipy import signal
from skimage.restoration import denoise_wavelet
from skimage.filters import threshold_otsu
from skimage.morphology import opening, closing, disk
from skimage.transform import resize
from skimage.metrics import structural_similarity as ssim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix, classification_report, roc_curve
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
import tensorflow_model_optimization as tfmot
warnings.filterwarnings("ignore")
SEED=42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# -------------------- CONFIGURATION --------------------
SAVE_DIR = Path("PAI_MobileNetV3_Portable")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

PAT_ROOT = Path("/content/PAT-public-data")
MENDELEY_ROOT = Path("/content/mendeley_pai")  # change to your extracted Mendeley folder

IMG_SIZE = 128
BATCH_SIZE = 8
VAE_BATCH = 8
RECON_EPOCHS = 20
VAE_EPOCHS = 25
CLS_EPOCHS = 30
LR_RECON = 1e-3
LR_VAE = 1e-3
LR_CLS = 2e-4
MAX_PAT_PAIRS = None
MAX_MENDELEY_IMAGES = None

print("Set MENDELEY_ROOT to the folder containing the extracted Mendeley dataset before running the data-loading cells.")

In [ ]:
# Download Database 1 directly from the public PAT repository
if not PAT_ROOT.exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/yqx7150/PAT-public-data.git", str(PAT_ROOT)], check=True)
print("PAT files:", len(list(PAT_ROOT.glob("*.mat"))))

In [ ]:
# -------------------- GENERAL NUMERIC / IMAGE HELPERS --------------------
def robust_minmax(x):
    x=np.asarray(x,dtype=np.float32)
    x=np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    lo,hi=np.percentile(x,[1,99])
    if hi<=lo: lo,hi=float(x.min()),float(x.max())
    x=np.clip(x,lo,hi)
    return ((x-lo)/(hi-lo+1e-8)).astype(np.float32)

def to_2d_numeric(mat_obj):
    candidates=[]
    for k,v in mat_obj.items():
        if k.startswith("__"): continue
        if isinstance(v,np.ndarray) and np.issubdtype(v.dtype,np.number):
            a=np.squeeze(v)
            if a.ndim>=2 and a.size>256:
                candidates.append((a.size,a))
    if not candidates:
        raise ValueError("No suitable numeric 2-D/3-D array was found in MAT file.")
    a=max(candidates,key=lambda z:z[0])[1]
    while a.ndim>2:
        a=np.mean(a,axis=-1)
    return robust_minmax(a)

def resize_img(x,size=IMG_SIZE):
    x=robust_minmax(x)
    return cv2.resize(x,(size,size),interpolation=cv2.INTER_AREA).astype(np.float32)

def ensure_3ch(x):
    x=np.asarray(x,dtype=np.float32)
    return np.repeat(x[...,None],3,axis=-1)

def show_grid(images,titles=None,n=4,figsize=(12,9)):
    images=list(images)[:n*n]
    rows=int(np.ceil(len(images)/n))
    plt.figure(figsize=figsize)
    for i,img in enumerate(images):
        plt.subplot(rows,n,i+1)
        plt.imshow(np.squeeze(img),cmap="gray")
        plt.axis("off")
        if titles: plt.title(titles[i],fontweight="bold")
    plt.tight_layout(); plt.show()

## Phase 2–4 — PAT preprocessing, portable acquisition simulation and lightweight U-Net reconstruction

For each compatible PAT phantom, the notebook pairs a lower-view acquisition (45 or 90 views) with its corresponding 180-view reference from the same phantom name. The lower-view input is filtered, wavelet-denoised, baseline-corrected and normalized before portable degradation. The 180-view reference is used only as the reconstruction target.

The pairing is based on the public file naming structure, not on a fabricated label.

In [ ]:
# -------------------- PAT PAIR DISCOVERY --------------------
mat_files=sorted(PAT_ROOT.glob("*.mat"))
by_key={}
for p in mat_files:
    m=re.match(r"phantom_sparse(45|90|180)_recon-(.+)\.mat$",p.name)
    if m:
        view=int(m.group(1)); key=m.group(2)
        by_key.setdefault(key,{})[view]=p

pairs=[]
for key,d in sorted(by_key.items()):
    if 180 in d:
        for low in [45,90]:
            if low in d:
                pairs.append((key,low,d[low],d[180]))
                break
if MAX_PAT_PAIRS: pairs=pairs[:MAX_PAT_PAIRS]
print("Reconstruction pairs:",len(pairs))
print(*[(k,v,a.name,t.name) for k,v,a,t in pairs],sep="\n")

In [ ]:
# -------------------- PAT SIGNAL PREPROCESSING --------------------
def butter_bandpass(x, low=0.01, high=0.45, order=4):
    x=np.asarray(x,dtype=np.float32)
    b,a=signal.butter(order,[low,high],btype="bandpass")
    if x.shape[-1] < 32:
        return x
    return signal.filtfilt(b,a,x,axis=-1).astype(np.float32)

def wavelet_denoise(x):
    try:
        y=denoise_wavelet(x,method="BayesShrink",mode="soft",wavelet="db2",rescale_sigma=True)
        return y.astype(np.float32)
    except Exception:
        return x.astype(np.float32)

def polynomial_baseline(x,degree=2):
    x=np.asarray(x,dtype=np.float32)
    if x.ndim==1:
        t=np.linspace(-1,1,len(x))
        return (x-np.polyval(np.polyfit(t,x,degree),t)).astype(np.float32)
    t=np.linspace(-1,1,x.shape[-1])
    out=np.empty_like(x)
    for i,row in enumerate(x.reshape(-1,x.shape[-1])):
        try: out.reshape(-1,x.shape[-1])[i]=row-np.polyval(np.polyfit(t,row,degree),t)
        except Exception: out.reshape(-1,x.shape[-1])[i]=row-row.mean()
    return out

def preprocess_pat(x):
    x=to_2d_numeric(x)
    x=butter_bandpass(x)
    x=wavelet_denoise(x)
    x=polynomial_baseline(x)
    return robust_minmax(x)

def portable_degrade(x,sparse_fraction=0.35,noise_std=0.035,limited_fraction=0.70):
    x=np.asarray(x,dtype=np.float32).copy()
    rng=np.random.default_rng(SEED)
    x=x+ rng.normal(0,noise_std,size=x.shape).astype(np.float32)
    keep=max(1,int(x.shape[0]*sparse_fraction))
    idx=np.sort(rng.choice(x.shape[0],keep,replace=False))
    sparse=np.zeros_like(x); sparse[idx]=x[idx]
    cols=max(1,int(sparse.shape[1]*limited_fraction))
    start=max(0,(sparse.shape[1]-cols)//2)
    limited=np.zeros_like(sparse)
    limited[:,start:start+cols]=sparse[:,start:start+cols]
    return robust_minmax(limited)

X_pat=[]; Y_pat=[]
for key,low,inp_path,tgt_path in pairs:
    inp=sio.loadmat(inp_path)
    tgt=sio.loadmat(tgt_path)
    xi=preprocess_pat(inp)
    yt=resize_img(to_2d_numeric(tgt))
    xi=resize_img(portable_degrade(xi))
    X_pat.append(xi[...,None]); Y_pat.append(yt[...,None])
X_pat=np.asarray(X_pat,np.float32); Y_pat=np.asarray(Y_pat,np.float32)
print("PAT input:",X_pat.shape,"PAT target:",Y_pat.shape)

In [ ]:
# -------------------- LIGHTWEIGHT U-NET --------------------
def ds_conv(x,filters,stride=1):
    x=layers.DepthwiseConv2D(3,strides=stride,padding="same",use_bias=False)(x)
    x=layers.BatchNormalization()(x); x=layers.Activation("relu")(x)
    x=layers.Conv2D(filters,1,padding="same",use_bias=False)(x)
    x=layers.BatchNormalization()(x); x=layers.Activation("relu")(x)
    return x

def residual_block(x,filters):
    skip=layers.Conv2D(filters,1,padding="same",use_bias=False)(x)
    y=ds_conv(x,filters)
    y=ds_conv(y,filters)
    return layers.Activation("relu")(layers.Add()([skip,y]))

def build_light_unet():
    inp=layers.Input((IMG_SIZE,IMG_SIZE,1))
    e1=residual_block(inp,16)
    p1=layers.MaxPooling2D()(e1)
    e2=residual_block(p1,32)
    p2=layers.MaxPooling2D()(e2)
    e3=residual_block(p2,64)
    p3=layers.MaxPooling2D()(e3)
    b=residual_block(p3,96)
    d3=layers.UpSampling2D(interpolation="bilinear")(b)
    d3=layers.Concatenate()([d3,e3]); d3=residual_block(d3,64)
    d2=layers.UpSampling2D(interpolation="bilinear")(d3)
    d2=layers.Concatenate()([d2,e2]); d2=residual_block(d2,32)
    d1=layers.UpSampling2D(interpolation="bilinear")(d2)
    d1=layers.Concatenate()([d1,e1]); d1=residual_block(d1,16)
    out=layers.Conv2D(1,1,activation="sigmoid")(d1)
    return Model(inp,out,name="Lightweight_PAT_UNet")

def l1_ssim_loss(y_true,y_pred):
    l1=tf.reduce_mean(tf.abs(y_true-y_pred))
    s=tf.reduce_mean(tf.image.ssim(y_true,y_pred,max_val=1.0))
    return 0.7*l1+0.3*(1.0-s)

unet=build_light_unet()
unet.compile(optimizer=keras.optimizers.Adam(LR_RECON),loss=l1_ssim_loss)
unet.summary()

In [ ]:
# -------------------- TRAIN / VALIDATE RECONSTRUCTION --------------------
if len(X_pat)>=2:
    idx=np.arange(len(X_pat))
    tr,va=train_test_split(idx,test_size=max(1,int(0.2*len(idx))),random_state=SEED)
    hist_recon=unet.fit(X_pat[tr],Y_pat[tr],validation_data=(X_pat[va],Y_pat[va]),
                        epochs=RECON_EPOCHS,batch_size=min(4,len(tr)),verbose=1)
    recon_pat=unet.predict(X_pat,batch_size=4,verbose=0)
    recon_metrics=[]
    for yt,yp in zip(Y_pat,recon_pat):
        mse=float(np.mean((yt-yp)**2))
        recon_metrics.append({"MSE":mse,"PSNR":float(tf.image.psnr(yt[None],yp[None],1.0)[0]),"SSIM":float(tf.image.ssim(yt[None],yp[None],1.0)[0])})
    print(pd.DataFrame(recon_metrics).mean(numeric_only=True))
else:
    raise RuntimeError("At least two PAT reconstruction pairs are required.")

In [ ]:
show_grid([X_pat[i,...,0] for i in range(min(4,len(X_pat)))],["Portable input"]*min(4,len(X_pat)),n=4)
show_grid([recon_pat[i,...,0] for i in range(min(4,len(recon_pat)))],["Reconstructed"]*min(4,len(recon_pat)),n=4)
show_grid([Y_pat[i,...,0] for i in range(min(4,len(Y_pat)))],["Reference"]*min(4,len(Y_pat)),n=4)

## Phase 5–6 — VAE denoising and ROI extraction

The VAE is trained unsupervised on images from both databases. Database 1 uses reconstructed PAT images; Database 2 uses the original vascular PAI images. The classifier labels are **not** used during VAE training.

In [ ]:
# -------------------- MENDELEY IMAGE DISCOVERY --------------------
if not MENDELEY_ROOT.exists():
    raise FileNotFoundError(f"Set MENDELEY_ROOT to the extracted Mendeley dataset: {MENDELEY_ROOT}")

image_exts={".png",".jpg",".jpeg",".bmp",".tif",".tiff",".webp"}
all_imgs=[p for p in MENDELEY_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in image_exts]

def infer_label(path):
    s=str(path).lower()
    if re.search(r"(^|[/_\\-])(tumou?r|tumor|cancer|glioma)([/_\\-]|$)",s): return 1
    if re.search(r"(^|[/_\\-])(normal|healthy|control|non[-_ ]?tumou?r)([/_\\-]|$)",s): return 0
    return None

m_rows=[{"path":str(p),"label":infer_label(p)} for p in all_imgs]
m_df=pd.DataFrame(m_rows)
print("Images:",len(m_df))
print("Label counts:",m_df["label"].value_counts(dropna=False).to_dict())
if m_df["label"].isna().any():
    print("Unlabelled files:",int(m_df["label"].isna().sum()))
    print(m_df[m_df.label.isna()].head(20).to_string(index=False))
    raise ValueError("Could not infer normal/tumour labels from the extracted dataset paths. Rename/place files in normal and tumor folders before continuing.")
m_df["label"]=m_df["label"].astype(int)
if MAX_MENDELEY_IMAGES: m_df=m_df.iloc[:MAX_MENDELEY_IMAGES].copy()
m_df.head()

In [ ]:
def load_gray(path):
    arr=np.array(Image.open(path).convert("L"),dtype=np.float32)/255.0
    return resize_img(arr)

X_m=np.asarray([load_gray(p) for p in m_df.path],np.float32)[...,None]
y_m=m_df.label.to_numpy(np.int32)
print(X_m.shape,y_m.shape)
print(pd.Series(y_m).value_counts().sort_index())

In [ ]:
# -------------------- VAE --------------------
def sampling(args):
    z_mean,z_log_var=args
    eps=tf.random.normal(shape=tf.shape(z_mean))
    return z_mean+tf.exp(0.5*z_log_var)*eps

def build_vae(latent_dim=64):
    inp=layers.Input((IMG_SIZE,IMG_SIZE,1))
    x=layers.Conv2D(32,3,2,padding="same",activation="relu")(inp)
    x=layers.Conv2D(64,3,2,padding="same",activation="relu")(x)
    x=layers.Conv2D(96,3,2,padding="same",activation="relu")(x)
    shape=tuple(x.shape[1:])
    x=layers.Flatten()(x)
    x=layers.Dense(128,activation="relu")(x)
    zm=layers.Dense(latent_dim,name="z_mean")(x)
    zv=layers.Dense(latent_dim,name="z_log_var")(x)
    z=layers.Lambda(sampling,name="z")([zm,zv])
    enc=Model(inp,[zm,zv,z],name="VAE_Encoder")
    zi=layers.Input((latent_dim,))
    x=layers.Dense(np.prod(shape),activation="relu")(zi)
    x=layers.Reshape(shape)(x)
    x=layers.Conv2DTranspose(96,3,2,padding="same",activation="relu")(x)
    x=layers.Conv2DTranspose(64,3,2,padding="same",activation="relu")(x)
    x=layers.Conv2DTranspose(32,3,2,padding="same",activation="relu")(x)
    out=layers.Conv2D(1,3,padding="same",activation="sigmoid")(x)
    dec=Model(zi,out,name="VAE_Decoder")
    class VAE(Model):
        def __init__(self,encoder,decoder):
            super().__init__(); self.encoder=encoder; self.decoder=decoder
        def train_step(self,data):
            if isinstance(data,tuple): data=data[0]
            with tf.GradientTape() as tape:
                zm,zv,z=self.encoder(data,training=True)
                rec=self.decoder(z,training=True)
                rec_loss=tf.reduce_mean(tf.abs(data-rec))
                kl=-0.5*tf.reduce_mean(1+zv-tf.square(zm)-tf.exp(zv))
                loss=rec_loss+1e-3*kl
            grads=tape.gradient(loss,self.trainable_weights)
            self.optimizer.apply_gradients(zip(grads,self.trainable_weights))
            return {"loss":loss,"reconstruction":rec_loss,"kl":kl}
    vae=VAE(enc,dec)
    vae.compile(optimizer=keras.optimizers.Adam(LR_VAE))
    return vae,enc,dec

vae,encoder,decoder=build_vae()
X_vae=np.concatenate([recon_pat,X_m],axis=0)
vae.fit(X_vae,epochs=VAE_EPOCHS,batch_size=VAE_BATCH,shuffle=True,verbose=1)
denoised_pat=decoder.predict(encoder.predict(recon_pat,batch_size=4,verbose=0)[2],batch_size=4,verbose=0)
denoised_m=decoder.predict(encoder.predict(X_m,batch_size=4,verbose=0)[2],batch_size=4,verbose=0)
print(denoised_pat.shape,denoised_m.shape)

In [ ]:
# -------------------- ROI EXTRACTION --------------------
def extract_roi(img):
    img=np.squeeze(img).astype(np.float32)
    img=robust_minmax(img)
    try: th=threshold_otsu(img)
    except: th=float(img.mean())
    mask=img>th
    mask=opening(mask,disk(2))
    mask=closing(mask,disk(3))
    ys,xs=np.where(mask)
    if len(xs)<20:
        return cv2.resize(img,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_LINEAR)
    x0,x1=max(0,xs.min()-4),min(img.shape[1],xs.max()+5)
    y0,y1=max(0,ys.min()-4),min(img.shape[0],ys.max()+5)
    roi=img[y0:y1,x0:x1]
    return cv2.resize(roi,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_LINEAR).astype(np.float32)

roi_pat=np.asarray([extract_roi(x) for x in denoised_pat],np.float32)[...,None]
roi_m=np.asarray([extract_roi(x) for x in denoised_m],np.float32)[...,None]
show_grid([roi_m[i,...,0] for i in range(min(8,len(roi_m)))],n=4)

## Phase 7 — Shared MobileNetV3-Small feature extraction

A single shared MobileNetV3-Small backbone is used for both databases. ImageNet weights are used when available; the first layer receives three replicated grayscale channels.

In [ ]:
# -------------------- SHARED MOBILENETV3-SMALL --------------------
backbone=tf.keras.applications.MobileNetV3Small(
    input_shape=(IMG_SIZE,IMG_SIZE,3),include_top=False,weights="imagenet",pooling=None
)
backbone.trainable=False

def mobile_input(x):
    x=np.asarray(x,np.float32)
    x=np.repeat(x,3,axis=-1)
    return tf.keras.applications.mobilenet_v3.preprocess_input(x*255.0)

pat_rgb=mobile_input(roi_pat)
m_rgb=mobile_input(roi_m)
F_pat=backbone.predict(pat_rgb,batch_size=8,verbose=0)
F_m=backbone.predict(m_rgb,batch_size=8,verbose=0)
print("Feature maps:",F_pat.shape,F_m.shape)

## Phase 8 — Cross-database feature fusion

Since there is no valid one-to-one correspondence between Database 1 and Database 2, a train-only Database-1 feature context is constructed. The context is spatially resized to the MobileNet feature-map size and broadcast to each Database-2 sample. Both representations are projected to the same channel dimension, z-score standardized using training statistics, concatenated, passed through coordinate attention, and compressed with a 1×1 convolution.

This is a valid cross-database fusion mechanism without fabricating paired patients.

In [ ]:
# -------------------- TRAIN / VAL / TEST SPLIT ON MENDELEY --------------------
idx=np.arange(len(m_df))
train_idx,temp_idx=train_test_split(idx,test_size=0.30,stratify=y_m,random_state=SEED)
val_idx,test_idx=train_test_split(temp_idx,test_size=0.50,stratify=y_m[temp_idx],random_state=SEED)
print(len(train_idx),len(val_idx),len(test_idx))
print("train",np.bincount(y_m[train_idx]),"val",np.bincount(y_m[val_idx]),"test",np.bincount(y_m[test_idx]))

In [ ]:
# Feature normalization statistics learned from training data only
pat_context=np.mean(F_pat,axis=0,keepdims=True)
pat_context=np.repeat(pat_context,len(F_m),axis=0)

# Channel-wise z-score across the spatial and sample dimensions.
mu=np.mean(np.concatenate([F_m[train_idx],np.repeat(F_pat[:1],len(train_idx),axis=0)],axis=0),axis=(0,1,2),keepdims=True)
sd=np.std(np.concatenate([F_m[train_idx],np.repeat(F_pat[:1],len(train_idx),axis=0)],axis=0),axis=(0,1,2),keepdims=True)+1e-6
F_m_z=(F_m-mu)/sd
F_pat_ctx_z=(pat_context-mu)/sd

F_m_z.shape,F_pat_ctx_z.shape

In [ ]:
# -------------------- COORDINATE ATTENTION + FUSION MODEL --------------------
def coord_attention(x,reduction=8):
    c=int(x.shape[-1])
    hpool=tf.reduce_mean(x,axis=2,keepdims=True)
    wpool=tf.reduce_mean(x,axis=1,keepdims=True)
    wpool=tf.transpose(wpool,[0,2,1,3])
    y=layers.Concatenate(axis=1)([hpool,wpool])
    y=layers.Conv2D(max(8,c//reduction),1,activation="relu",padding="same")(y)
    h,w=tf.shape(x)[1],tf.shape(x)[2]
    yh,yw=tf.split(y,[h,w],axis=1)
    yw=tf.transpose(yw,[0,2,1,3])
    ah=layers.Conv2D(c,1,activation="sigmoid")(yh)
    aw=layers.Conv2D(c,1,activation="sigmoid")(yw)
    return x*ah*aw

inp=layers.Input(F_m_z.shape[1:],name="Mendeley_feature")
ctx=layers.Input(F_pat_ctx_z.shape[1:],name="PAT_context_feature")
a=layers.Conv2D(64,1,padding="same",activation="relu")(inp)
b=layers.Conv2D(64,1,padding="same",activation="relu")(ctx)
f=layers.Concatenate()([a,b])
f=coord_attention(f)
f=layers.Conv2D(64,1,padding="same",activation="relu",name="fusion_1x1")(f)
f=layers.BatchNormalization()(f)
f=layers.GlobalAveragePooling2D()(f)
f=layers.Dropout(0.25)(f)
f=layers.Dense(64,activation="relu")(f)
f=layers.Dropout(0.20)(f)
out=layers.Dense(2,activation="softmax",name="class_output")(f)
classifier=Model([inp,ctx],out,name="CrossDatabase_MobileNetV3_Fusion")
classifier.compile(optimizer=keras.optimizers.Adam(LR_CLS),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
classifier.summary()

In [ ]:
# -------------------- CLASSIFIER TRAINING --------------------
callbacks=[
    keras.callbacks.EarlyStopping(monitor="val_loss",patience=7,restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss",factor=0.5,patience=3,min_lr=1e-6)
]
hist_cls=classifier.fit(
    [F_m_z[train_idx],F_pat_ctx_z[train_idx]],y_m[train_idx],
    validation_data=([F_m_z[val_idx],F_pat_ctx_z[val_idx]],y_m[val_idx]),
    epochs=CLS_EPOCHS,batch_size=BATCH_SIZE,callbacks=callbacks,verbose=1
)

In [ ]:
# -------------------- FINAL TEST METRICS --------------------
probs=classifier.predict([F_m_z[test_idx],F_pat_ctx_z[test_idx]],verbose=0)
pred=np.argmax(probs,axis=1)
yt=y_m[test_idx]
acc=accuracy_score(yt,pred)
bal=balanced_accuracy_score(yt,pred)
prec,rec,f1,_=precision_recall_fscore_support(yt,pred,average="binary",zero_division=0)
auc=roc_auc_score(yt,probs[:,1]) if len(np.unique(yt))==2 else np.nan
metrics=pd.DataFrame([{"Accuracy":acc,"Balanced_Accuracy":bal,"Precision":prec,"Recall":rec,"F1":f1,"AUROC":auc}])
print(metrics.to_string(index=False))
print(classification_report(yt,pred,target_names=["Normal","Tumour"],zero_division=0))

cm=confusion_matrix(yt,pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",xticklabels=["Normal","Tumour"],yticklabels=["Normal","Tumour"])
plt.xlabel("Predicted",fontweight="bold"); plt.ylabel("Actual",fontweight="bold"); plt.tight_layout()
plt.savefig(SAVE_DIR/"confusion_matrix.png",dpi=600,bbox_inches="tight"); plt.show()

fpr,tpr,_=roc_curve(yt,probs[:,1])
plt.figure(figsize=(7,6)); plt.plot(fpr,tpr,label=f"MobileNetV3 Fusion AUROC={auc:.4f}")
plt.plot([0,1],[0,1],"--"); plt.xlabel("False Positive Rate",fontweight="bold"); plt.ylabel("True Positive Rate",fontweight="bold")
plt.legend(); plt.tight_layout(); plt.savefig(SAVE_DIR/"roc_curve.png",dpi=600,bbox_inches="tight"); plt.show()

## Phase 10 — Edge model compression

The deployment model is kept TensorFlow/Keras-compatible first. A structured channel-pruning stage is applied to the compact fusion head using TensorFlow Model Optimization, followed by short fine-tuning and INT8 TensorFlow Lite conversion.

The representative dataset is taken only from the training partition.

In [ ]:
# -------------------- STRUCTURED PRUNING --------------------
pruning_params={
    "pruning_schedule": tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.10,final_sparsity=0.50,begin_step=0,end_step=max(1,len(train_idx)//BATCH_SIZE*5)
    )
}
pruned= tfmot.sparsity.keras.prune_low_magnitude(classifier,**pruning_params)
pruned.compile(optimizer=keras.optimizers.Adam(5e-5),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
pruned.fit([F_m_z[train_idx],F_pat_ctx_z[train_idx]],y_m[train_idx],
           validation_data=([F_m_z[val_idx],F_pat_ctx_z[val_idx]],y_m[val_idx]),
           epochs=5,batch_size=BATCH_SIZE,
           callbacks=[tfmot.sparsity.keras.UpdatePruningStep()],verbose=1)
stripped=tfmot.sparsity.keras.strip_pruning(pruned)
stripped.save(SAVE_DIR/"compressed_keras.keras")
print("Saved:",SAVE_DIR/"compressed_keras.keras")

In [ ]:
# -------------------- INT8 TFLITE CONVERSION --------------------
def representative_data():
    n=min(100,len(train_idx))
    for i in train_idx[:n]:
        yield [
            F_m_z[i:i+1].astype(np.float32),
            F_pat_ctx_z[i:i+1].astype(np.float32)
        ]

converter=tf.lite.TFLiteConverter.from_keras_model(stripped)
converter.optimizations=[tf.lite.Optimize.DEFAULT]
converter.representative_dataset=representative_data
converter.target_spec.supported_ops=[tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type=tf.int8
converter.inference_output_type=tf.int8
tflite_model=converter.convert()
with open(SAVE_DIR/"PAI_MobileNetV3_Fusion_INT8.tflite","wb") as f: f.write(tflite_model)
print("TFLite size (KB):",round(len(tflite_model)/1024,2))

## Phase 11 — Confidence-based decision

For every prediction:

- confidence = maximum class probability;
- predictive entropy = normalized Shannon entropy of the two-class posterior;
- high-confidence prediction requires confidence ≥ threshold and entropy ≤ threshold;
- otherwise the case is flagged for expert review.

The thresholds are fixed configuration values rather than tuned on the test set.

In [ ]:
# -------------------- CONFIDENCE / ENTROPY --------------------
CONF_THRESHOLD=0.80
ENTROPY_THRESHOLD=0.50

def confidence_decision(prob,conf_thr=CONF_THRESHOLD,entropy_thr=ENTROPY_THRESHOLD):
    prob=np.asarray(prob,dtype=np.float64)
    prob=np.clip(prob,1e-8,1)
    conf=float(np.max(prob))
    entropy=float(-np.sum(prob*np.log(prob))/np.log(len(prob)))
    pred=int(np.argmax(prob))
    status="High-confidence" if conf>=conf_thr and entropy<=entropy_thr else "Expert-review"
    return pred,conf,entropy,status

decision_rows=[]
for i,p in zip(test_idx,probs):
    pred_i,conf_i,ent_i,status=confidence_decision(p)
    decision_rows.append({
        "file":m_df.iloc[i].path,
        "true_class":"Tumour" if y_m[i]==1 else "Normal",
        "predicted_class":"Tumour" if pred_i==1 else "Normal",
        "confidence":conf_i,
        "predictive_entropy":ent_i,
        "decision":status
    })
decision_df=pd.DataFrame(decision_rows)
display(decision_df)
decision_df.to_csv(SAVE_DIR/"confidence_decisions.csv",index=False)

In [ ]:
# -------------------- SAVE MODELS, FEATURES AND RESULTS --------------------
np.save(SAVE_DIR/"PAT_reconstructed_images.npy",recon_pat)
np.save(SAVE_DIR/"Mendeley_denoised_images.npy",denoised_m)
np.save(SAVE_DIR/"Mendeley_ROI_images.npy",roi_m)
np.save(SAVE_DIR/"Mendeley_labels.npy",y_m)
np.save(SAVE_DIR/"PAT_feature_maps.npy",F_pat)
np.save(SAVE_DIR/"Mendeley_feature_maps.npy",F_m)
metrics.to_csv(SAVE_DIR/"classification_metrics.csv",index=False)
with open(SAVE_DIR/"pipeline_config.json","w") as f:
    json.dump({
        "seed":SEED,"image_size":IMG_SIZE,"reconstruction_epochs":RECON_EPOCHS,
        "vae_epochs":VAE_EPOCHS,"classification_epochs":CLS_EPOCHS,
        "confidence_threshold":CONF_THRESHOLD,"entropy_threshold":ENTROPY_THRESHOLD
    },f,indent=2)
unet.save(SAVE_DIR/"lightweight_unet_reconstruction.keras")
encoder.save(SAVE_DIR/"vae_encoder.keras")
decoder.save(SAVE_DIR/"vae_decoder.keras")
classifier.save(SAVE_DIR/"cross_database_mobilenetv3_fusion.keras")
print("All artifacts saved to:",SAVE_DIR.resolve())

## End-to-end output

The notebook produces:

1. `lightweight_unet_reconstruction.keras`
2. `PAT_reconstructed_images.npy`
3. `vae_encoder.keras`
4. `vae_decoder.keras`
5. `Mendeley_denoised_images.npy`
6. `Mendeley_ROI_images.npy`
7. `cross_database_mobilenetv3_fusion.keras`
8. `compressed_keras.keras`
9. `PAI_MobileNetV3_Fusion_INT8.tflite`
10. `classification_metrics.csv`
11. `confusion_matrix.png`
12. `roc_curve.png`
13. `confidence_decisions.csv`

### Experimental-design constraint

Database 1 is a reconstruction/pretraining source and does not receive invented normal/tumour labels. Database 2 supplies the normal/tumour classification labels. The final model therefore measures tumour-associated vascular abnormality detection on the labelled vascular dataset while using PAT data to learn portable reconstruction and cross-database representation context.